# Container Lifecycle & Management

Notebook 04 covered `docker run` and the basics of starting, stopping, and removing containers. This notebook goes deeper — restart policies, signals and graceful shutdown, real-time monitoring, bulk cleanup, and managing containers at scale.

Topics:
- Full lifecycle state machine
- Restart policies: `no`, `always`, `on-failure`, `unless-stopped`
- Signals and graceful shutdown — what `docker stop` actually does
- `docker stats` — real-time resource monitoring
- `docker events` — the daemon event stream
- `docker update` — change config on a running container
- Filtering and formatting output
- `docker system prune` and targeted cleanup
- Container naming and labelling strategies

## 1. The Full Lifecycle State Machine

A container moves through well-defined states. Understanding the transitions tells you which commands apply at each stage.

```
                    ┌─────────────────────────────────┐
                    │                                 │
           docker create                              │
                    │                           docker start
                    ▼                                 │
              [created] ─── docker start ──► [running] ◄───────────────┐
                                                 │                      │
                                         docker pause              docker restart
                                                 │                      │
                                                 ▼                      │
                                           [paused]                     │
                                                 │                      │
                                         docker unpause                 │
                                                 │                      │
                                    docker stop / kill                  │
                                     (or process exits)                 │
                                                 │                      │
                                                 ▼                      │
                                           [exited] ────────────────────┘
                                                 │
                                           docker rm
                                                 │
                                                 ▼
                                           [deleted]
```

| State | Description |
|-------|-------------|
| `created` | Container allocated, writable layer created, not yet started |
| `running` | PID 1 is running |
| `paused` | All processes frozen via cgroup freeze (SIGSTOP equivalent) |
| `restarting` | Container is in the middle of a restart (briefly visible) |
| `exited` | PID 1 has exited — container exists but is not running |
| `dead` | Container removal failed — rare, usually a storage issue |

**Key insight:** An exited container still occupies disk space (writable layer + logs). It must be explicitly removed with `docker rm` — or you accumulate stopped containers indefinitely.

## 2. Signals and Graceful Shutdown

When you run `docker stop`, Docker does **not** immediately kill the container. It sends a signal and waits.

```
docker stop CONTAINER
    │
    ├─ Send SIGTERM to PID 1     ← "please shut down gracefully"
    │   (default signal, configurable with STOPSIGNAL in Dockerfile)
    │
    ├─ Wait up to 10 seconds     ← configurable with --time / -t flag
    │
    └─ Send SIGKILL if still running  ← forced termination
```

### Why graceful shutdown matters

A web server that receives SIGTERM should:
1. Stop accepting new connections
2. Finish handling in-flight requests
3. Flush logs and close file handles
4. Exit cleanly

If you use shell form CMD (`CMD gunicorn app:app` instead of `CMD ["gunicorn", "app:app"]`), the shell becomes PID 1 and swallows SIGTERM without forwarding it to your process — graceful shutdown breaks silently.

### `STOPSIGNAL`

You can configure which signal Docker sends in the Dockerfile:

```dockerfile
# Nginx expects SIGQUIT for graceful drain (not SIGTERM)
STOPSIGNAL SIGQUIT
```

Or at runtime:

```bash
docker stop --signal SIGQUIT mycontainer
docker kill --signal SIGHUP mycontainer   # reload config without restart
```

In [ ]:
# Demonstrate graceful vs forced shutdown timing
# A container that sleeps for 60s — simulates a long-running process

# Start it
!docker run -d --name lifecycle-demo alpine sleep 60

import time, subprocess

# docker stop with short timeout — SIGTERM then SIGKILL after 2s
t0 = time.time()
subprocess.run(["docker", "stop", "--time", "2", "lifecycle-demo"])
elapsed = time.time() - t0
print(f"docker stop took {elapsed:.1f}s (2s grace + overhead)")

# docker kill is immediate — no grace period
!docker run -d --name lifecycle-kill alpine sleep 60
t0 = time.time()
subprocess.run(["docker", "kill", "lifecycle-kill"])
elapsed = time.time() - t0
print(f"docker kill took {elapsed:.2f}s (immediate)")

!docker rm lifecycle-demo lifecycle-kill

## 3. Restart Policies

A restart policy tells Docker what to do when a container's PID 1 exits. Set it with `--restart` in `docker run`.

| Policy | Behaviour |
|--------|----------|
| `no` | Never restart (default) |
| `always` | Always restart, regardless of exit code. Also restarts when the Docker daemon starts (survives reboots). |
| `on-failure[:max]` | Restart only if exit code is non-zero. Optional max retry count. |
| `unless-stopped` | Like `always` but does **not** restart if the container was manually stopped before the daemon restarted. |

### When to use which

```
Production long-running service?  → unless-stopped or always
Worker that might legitimately fail?  → on-failure:5
One-shot task?  → no (default)
Critical infrastructure (database)?  → always
```

**`always` vs `unless-stopped`:** If you manually run `docker stop mydb`, then reboot the host:
- `always` — `mydb` restarts on boot even though you explicitly stopped it
- `unless-stopped` — `mydb` stays stopped, respecting your manual intervention

In [ ]:
# on-failure demo: container exits with code 1, Docker restarts it
# max 3 retries — after that it stays stopped
!docker run -d --name restart-demo \
    --restart on-failure:3 \
    alpine sh -c 'echo "attempt"; exit 1'

import time
time.sleep(4)  # let it retry a few times

# RestartCount shows how many times Docker has restarted the container
!docker inspect restart-demo \
    --format 'Status: {{.State.Status}}  ExitCode: {{.State.ExitCode}}  Restarts: {{.RestartCount}}'

!docker rm -f restart-demo

In [ ]:
# A container that exits cleanly (exit 0) is NOT restarted by on-failure
!docker run -d --name exit-zero \
    --restart on-failure:3 \
    alpine sh -c 'echo "done"; exit 0'

time.sleep(2)
!docker inspect exit-zero \
    --format 'Status: {{.State.Status}}  ExitCode: {{.State.ExitCode}}  Restarts: {{.RestartCount}}'

!docker rm exit-zero

## 4. `docker update` — Change Config Without Recreating

`docker update` lets you modify certain container settings without stopping and recreating the container. Useful for adjusting resource limits in response to load.

```bash
# Change the restart policy on a running container
docker update --restart unless-stopped mycontainer

# Adjust memory limit
docker update --memory 512m --memory-swap 1g mycontainer

# Adjust CPU limit
docker update --cpus 1.5 mycontainer

# Apply to multiple containers at once
docker update --restart unless-stopped web db cache
```

What can be updated on a running container: restart policy, memory/CPU limits, blkio weights. What cannot: port mappings, volumes, environment variables, the image.

In [ ]:
# Start with no restart policy, then update it
!docker run -d --name updatable nginx:alpine

!docker inspect updatable --format 'RestartPolicy: {{.HostConfig.RestartPolicy.Name}}'

!docker update --restart unless-stopped updatable

!docker inspect updatable --format 'RestartPolicy: {{.HostConfig.RestartPolicy.Name}}'

!docker rm -f updatable

## 5. Real-Time Monitoring with `docker stats`

`docker stats` streams live resource usage metrics for running containers — CPU, memory, network I/O, and block I/O.

```bash
docker stats                   # all running containers, live stream
docker stats mycontainer       # one container
docker stats --no-stream       # one snapshot, then exit (good for scripts)
docker stats --format "table {{.Name}}\t{{.CPUPerc}}\t{{.MemUsage}}"
```

Output columns:

| Column | Meaning |
|--------|---------|
| `CPU %` | CPU usage as a percentage of the allocated CPUs |
| `MEM USAGE / LIMIT` | Current memory used vs the container's memory limit |
| `MEM %` | Memory usage as a percentage of the limit |
| `NET I/O` | Bytes sent and received on the container's network interface |
| `BLOCK I/O` | Bytes read from and written to block devices |
| `PIDS` | Number of processes inside the container |

In [ ]:
# Start a container to monitor
!docker run -d --name stats-demo --memory 128m nginx:alpine

import time; time.sleep(1)

# --no-stream: print one snapshot and exit (works well in notebooks)
!docker stats stats-demo --no-stream \
    --format 'table {{.Name}}\t{{.CPUPerc}}\t{{.MemUsage}}\t{{.MemPerc}}\t{{.PIDs}}'

!docker rm -f stats-demo

## 6. `docker events` — The Daemon Event Stream

`docker events` streams real-time events from the Docker daemon — container lifecycle, image pulls, network and volume operations. Essential for debugging, auditing, and building automation.

```bash
docker events                                    # stream all events
docker events --filter type=container            # containers only
docker events --filter event=start               # start events only
docker events --filter container=myapp           # one container
docker events --since 5m                         # last 5 minutes
docker events --since 2025-04-01T00:00:00 --until 2025-04-01T01:00:00
```

Common event types: `create`, `start`, `stop`, `kill`, `die`, `destroy`, `pause`, `unpause`, `restart`, `pull`, `push`, `connect`, `disconnect`.

In [ ]:
import subprocess, threading, time

# Collect events in the background while we run a container
events = []

def collect_events():
    proc = subprocess.Popen(
        ["docker", "events", "--filter", "container=events-demo",
         "--filter", "type=container", "--format", "{{.Time}} {{.Action}}"],
        stdout=subprocess.PIPE, text=True
    )
    for line in proc.stdout:
        events.append(line.strip())
        if len(events) >= 4:   # stop after collecting 4 events
            proc.terminate()
            break

t = threading.Thread(target=collect_events, daemon=True)
t.start()

time.sleep(0.3)  # give events listener a moment to attach

# Generate lifecycle events
subprocess.run(["docker", "run", "-d", "--name", "events-demo", "alpine", "sleep", "5"],
               capture_output=True)
time.sleep(0.2)
subprocess.run(["docker", "pause",   "events-demo"], capture_output=True)
time.sleep(0.2)
subprocess.run(["docker", "unpause", "events-demo"], capture_output=True)
time.sleep(0.2)
subprocess.run(["docker", "stop",    "events-demo"], capture_output=True)

t.join(timeout=5)

print("Events captured:")
for e in events:
    print(" ", e)

!docker rm events-demo

## 7. Filtering and Formatting `docker ps`

On hosts running many containers, raw `docker ps` output becomes unwieldy. Filters and custom formats help.

```bash
# All containers including stopped
docker ps -a

# Filter by status
docker ps --filter status=exited
docker ps --filter status=running

# Filter by name (substring match)
docker ps --filter name=web

# Filter by label
docker ps --filter label=env=production

# Filter by ancestor image
docker ps --filter ancestor=nginx:alpine

# Custom format — show only specific columns
docker ps --format 'table {{.Names}}\t{{.Status}}\t{{.Ports}}'

# IDs only — useful for piping to other commands
docker ps -q                          # running container IDs
docker ps -aq                         # all container IDs (including stopped)
docker ps -aq --filter status=exited  # exited container IDs
```

In [ ]:
# Start containers with labels for filtering
!docker run -d --name web-prod  --label env=production --label tier=web  nginx:alpine
!docker run -d --name web-stage --label env=staging    --label tier=web  nginx:alpine
!docker run -d --name db-prod   --label env=production --label tier=db   alpine sleep 60

print("All running:")
!docker ps --format 'table {{.Names}}\t{{.Status}}\t{{.Labels}}' \
    --filter name=prod --filter name=stage 2>/dev/null || \
    docker ps --format 'table {{.Names}}\t{{.Status}}' \
    --filter label=env

In [ ]:
# Filter by label
print("Production containers only:")
!docker ps --filter label=env=production \
    --format 'table {{.Names}}\t{{.Status}}'

print("\nWeb tier only:")
!docker ps --filter label=tier=web \
    --format 'table {{.Names}}\t{{.Status}}'

In [ ]:
# Stop all production containers using label filter + -q
print("Production container IDs:")
!docker ps -q --filter label=env=production

# Stop them all in one command
!docker stop $(docker ps -q --filter label=env=production) 2>/dev/null || true

print("After stop:")
!docker ps --filter label=env --format 'table {{.Names}}\t{{.Status}}'

In [ ]:
# Clean up all demo containers
!docker rm -f web-prod web-stage db-prod 2>/dev/null || true
!echo "Cleaned up"

## 8. Cleanup: Reclaiming Disk Space

Docker accumulates disk usage over time: stopped containers, unused images, orphaned volumes, and build cache. Several commands reclaim space.

### Targeted cleanup

```bash
# Remove all stopped containers
docker container prune

# Remove dangling images (untagged, not referenced by any container)
docker image prune

# Remove all unused images (not just dangling — any image not used by a running container)
docker image prune -a

# Remove unused volumes
docker volume prune

# Remove unused networks
docker network prune

# Remove build cache
docker builder prune
```

### Nuclear option: `docker system prune`

```bash
# Remove stopped containers + dangling images + unused networks + build cache
docker system prune

# Also remove unused volumes (add -v)
docker system prune -v

# Remove ALL unused images, not just dangling (add -a)
docker system prune -a

# Skip the confirmation prompt
docker system prune -f
```

Check disk usage before pruning:

```bash
docker system df          # summary
docker system df -v       # verbose, per-object breakdown
```

In [ ]:
# Check current Docker disk usage
!docker system df

In [ ]:
# Create some mess: run-and-exit containers without --rm
for i in range(5):
    !docker run alpine echo "container {i}" > /dev/null 2>&1

print("Stopped containers:")
!docker ps -a --filter status=exited --format 'table {{.Names}}\t{{.Status}}' | head -8

In [ ]:
# Remove only stopped containers (safe — doesn't touch running ones)
!docker container prune -f

print("After prune:")
!docker ps -a --filter status=exited --format 'table {{.Names}}\t{{.Status}}' | head -5
!docker system df

## 9. Naming and Labelling Strategies

On any non-trivial deployment, naming and labelling containers makes management far easier.

### Naming conventions

```bash
# Pattern: service-environment-instance
docker run -d --name api-prod-1     myapp:prod
docker run -d --name api-prod-2     myapp:prod
docker run -d --name worker-prod-1  myworker:prod
docker run -d --name postgres-prod  postgres:16
```

### Labels as queryable metadata

Labels let you attach arbitrary key-value metadata — then filter, manage, and monitor by it.

```bash
docker run -d \
    --name api-prod \
    --label com.example.service=api \
    --label com.example.env=production \
    --label com.example.version=2.3.1 \
    --label com.example.team=backend \
    myapp:2.3.1
```

Recommended label namespace: use reverse-DNS prefix (`com.yourcompany.`) to avoid clashes with Docker's own labels.

You can also set labels in the Dockerfile — they apply to every container run from that image:

```dockerfile
LABEL com.example.service="api" \
      com.example.version="2.3.1"
```

## 10. `docker inspect` for Deep Inspection

`docker inspect` returns the full JSON configuration and state of any Docker object — container, image, network, or volume.

In [ ]:
import subprocess, json

!docker run -d --name inspect-demo \
    --restart unless-stopped \
    --memory 64m \
    --label com.example.service=demo \
    -p 8090:80 \
    nginx:alpine

raw = subprocess.check_output(["docker", "inspect", "inspect-demo"])
data = json.loads(raw)[0]

# Extract commonly needed fields
print(f"Image:          {data['Config']['Image']}")
print(f"Status:         {data['State']['Status']}")
print(f"PID:            {data['State']['Pid']}")
print(f"RestartPolicy:  {data['HostConfig']['RestartPolicy']['Name']}")
print(f"Memory limit:   {data['HostConfig']['Memory'] // 1024**2} MB")
print(f"Labels:         {data['Config']['Labels']}")
print(f"Ports:          {data['NetworkSettings']['Ports']}")
print(f"IP address:     {data['NetworkSettings']['IPAddress']}")

!docker rm -f inspect-demo

## Summary

- **Container states:** `created` → `running` → `paused` / `exited` → deleted. Exited containers still consume disk until `docker rm`.
- **`docker stop` sends SIGTERM**, waits (default 10s), then SIGKILL. Use exec-form `CMD` so your process is PID 1 and receives SIGTERM correctly. Configure custom signals with `STOPSIGNAL`.
- **Restart policies:** `no` (default), `on-failure[:N]` (non-zero exit only), `always` (always, survives daemon restart), `unless-stopped` (always except when manually stopped).
- **`docker update`** changes restart policy and resource limits on running containers without recreating them.
- **`docker stats --no-stream`** gives a one-shot resource snapshot per container — CPU, memory, network, block I/O.
- **`docker events`** streams real-time daemon events — filter by container, event type, or time range.
- **Filtering:** use `--filter status=`, `--filter label=`, `--filter name=` on `docker ps`. Use `-q` to get IDs for scripting.
- **Cleanup:** `docker container prune` (stopped containers), `docker image prune [-a]` (unused images), `docker system prune` (everything unused). Check usage first with `docker system df`.
- **Labels** make containers queryable metadata — use reverse-DNS namespace, set in Dockerfile or at `docker run`.